# Data Mining Assignment — Apache Spark Data Lake (Bronze → Silver)**Name:** Kritika Yadav**Roll Number:** LCB2023006**Dataset:** `sales.csv`**Environment:** Google Colab + PySpark---### ObjectiveBuild a minimal **Data Lake** on top of `sales.csv`:| Layer | Contents | Rule ||---|---|---|| **Bronze** | the raw file, byte-for-byte as provided | never modified || **Silver** | the same records after *basic* PySpark preprocessing | written as Parquet + CSV |No Gold layer and no business analysis — the scope is loading, discrepancydetection, and basic cleaning.### Guiding principleEvery discrepancy count in this notebook is **computed by a Spark operation**,never typed in by hand. Cleaning is deliberately conservative: the aim is to*repair and keep* records, not to delete every unusual row. Each repair a rowreceives is recorded in a `dq_flags` column so nothing is silently changed.

## 0. Environment setup

In [1]:
# PySpark is not pre-installed in Colab.
!pip install -q pyspark==3.5.1

In [2]:
import os, glob, shutil, hashlib, urllib.request

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

spark = (
    SparkSession.builder
    .appName("SalesDataLake-Bronze-to-Silver")
    .config("spark.sql.session.timeZone", "UTC")
    # CORRECTED makes to_date() return NULL for impossible dates (e.g. 31 Feb)
    # instead of silently rolling them over to a valid date.
    .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/01 23:28:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.1


## 1. Data Lake structure```data_lake/├── bronze/│   └── sales.csv                  <- raw, untouched└── silver/    └── preprocessed_sales/        <- Spark output (Parquet)    └── preprocessed_sales_csv/    <- Spark output (CSV part files)```

In [3]:
BRONZE_DIR = "data_lake/bronze"
SILVER_DIR = "data_lake/silver"
SILVER_PARQUET = f"{SILVER_DIR}/preprocessed_sales"
SILVER_CSV_DIR = f"{SILVER_DIR}/preprocessed_sales_csv"
BRONZE_FILE = f"{BRONZE_DIR}/sales.csv"

os.makedirs(BRONZE_DIR, exist_ok=True)
os.makedirs(SILVER_DIR, exist_ok=True)

# Land the raw dataset in Bronze. Prefer a local copy (e.g. uploaded to Colab or
# sitting next to the notebook); otherwise pull it from the class repository so
# the notebook runs top-to-bottom unattended.
RAW_URL = ("https://raw.githubusercontent.com/necromancersanju-netizen/"
           "data-mining-data-lake-assignment/main/dataset/sales.csv")
LOCAL_CANDIDATES = ["sales.csv", "dataset/sales.csv", "../dataset/sales.csv",
                    "../../dataset/sales.csv", "/content/sales.csv"]

if not os.path.exists(BRONZE_FILE):
    src = next((p for p in LOCAL_CANDIDATES if os.path.exists(p)), None)
    if src:
        shutil.copyfile(src, BRONZE_FILE)
        print(f"Bronze loaded from local file: {src}")
    else:
        urllib.request.urlretrieve(RAW_URL, BRONZE_FILE)
        print("Bronze loaded from class repository URL")
else:
    print("Bronze file already present")

print("size:", os.path.getsize(BRONZE_FILE), "bytes")

Bronze loaded from local file: ../../dataset/sales.csv
size: 109537 bytes


In [4]:
# Fingerprint the Bronze file now, and re-check it at the end of the notebook to
# prove the raw layer was never written to.
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 16), b""):
            h.update(chunk)
    return h.hexdigest()

BRONZE_SHA_BEFORE = sha256(BRONZE_FILE)
print("bronze sha256:", BRONZE_SHA_BEFORE)
print("\nfirst 3 raw lines:")
with open(BRONZE_FILE) as f:
    for _ in range(3):
        print("   ", f.readline().rstrip())

bronze sha256: 1b2c7e0acea6250c992b890975ed0947b5bececa6a2880b707e4523e0ee22eb8

first 3 raw lines:
    order_id,customer_id,customer_name,product,category,quantity,unit_price,discount_percent,payment_method,city,state,order_date,order_status
    100001,C0256,Keerthi Mishra,Saree,Clothing,2,6710,20,Credit Card,Mumbai,Maharashtra,2026-01-01,Delivered
    100002,C0042,Arjun Singh,T-Shirt,Clothing,6,680,5,Debit Card,Chennai,Tamil Nadu,2025-08-03,Delivered


## 2. Load and inspect with SparkThe file is read **twice**, on purpose:1. `inferSchema=True` — to see what Spark *guesses* the types are.2. `inferSchema=False` (everything as `string`) — this is the DataFrame actually   used for the work. Reading raw text means a value like `not-a-date` or a   malformed number arrives intact and can be *counted* as a discrepancy, instead   of being turned into a `NULL` by the reader before it is ever seen.

In [5]:
inferred = spark.read.csv(BRONZE_FILE, header=True, inferSchema=True)
print("Schema as inferred by Spark:")
inferred.printSchema()

Schema as inferred by Spark:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)



In [6]:
raw = spark.read.csv(BRONZE_FILE, header=True, inferSchema=False)  # all StringType
raw.cache()

TOTAL_ROWS = raw.count()
print("Schema used for discrepancy detection (all columns read as raw text):")
raw.printSchema()
print(f"Total records in Bronze: {TOTAL_ROWS}")
print(f"Total columns: {len(raw.columns)}")

Schema used for discrepancy detection (all columns read as raw text):
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_percent: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

Total records in Bronze: 1000
Total columns: 13


In [7]:
raw.show(10, truncate=False)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12              |UPI             |P

### 2.1 Discrepancy detectionFive families of problem are looked for: duplicates, missing values,text-format inconsistencies, invalid numbers, and invalid dates.

In [8]:
# --- Duplicates ---------------------------------------------------------------
distinct_rows = raw.distinct().count()
exact_dupes = TOTAL_ROWS - distinct_rows

distinct_ids = raw.select("order_id").distinct().count()
dupe_id_rows = TOTAL_ROWS - distinct_ids

print(f"Total rows                         : {TOTAL_ROWS}")
print(f"Fully distinct rows                : {distinct_rows}")
print(f"Exact duplicate rows (removable)   : {exact_dupes}")
print(f"Rows sharing an order_id           : {dupe_id_rows}")

print("\norder_id values appearing more than once:")
(raw.groupBy("order_id").count()
    .filter("count > 1")
    .orderBy(F.desc("count"), "order_id")
    .show(25, truncate=False))

Total rows                         : 1000
Fully distinct rows                : 990
Exact duplicate rows (removable)   : 10
Rows sharing an order_id           : 18

order_id values appearing more than once:
+--------+-----+
|order_id|count|
+--------+-----+
|100055  |2    |
|100064  |2    |
|100072  |2    |
|100145  |2    |
|100297  |2    |
|100305  |2    |
|100317  |2    |
|100329  |2    |
|100331  |2    |
|100434  |2    |
|100444  |2    |
|100474  |2    |
|100502  |2    |
|100588  |2    |
|100631  |2    |
|100666  |2    |
|100692  |2    |
|100773  |2    |
+--------+-----+



In [9]:
# --- Missing values -----------------------------------------------------------
# An empty string and a whitespace-only string are as missing as a NULL, so all
# three are counted together.
def missing_expr(c):
    col = F.col(c)
    return F.sum(
        (col.isNull() | (F.trim(col) == F.lit(""))).cast("int")
    ).alias(c)

missing_counts = raw.select(*[missing_expr(c) for c in raw.columns]).collect()[0].asDict()

print("Missing / blank values per column:")
for c, n in missing_counts.items():
    if n:
        print(f"  {c:<18} {n:>4}  ({100 * n / TOTAL_ROWS:.1f}%)")
print("  (columns not listed have no missing values)")

Missing / blank values per column:
  customer_name        10  (1.0%)
  payment_method        8  (0.8%)
  city                  8  (0.8%)
  (columns not listed have no missing values)


In [10]:
# --- Text-format inconsistencies: stray whitespace ----------------------------
TEXT_COLS = ["customer_id", "customer_name", "product", "category",
             "payment_method", "city", "state", "order_status"]

ws_counts = raw.select(*[
    F.sum((F.col(c) != F.trim(F.col(c))).cast("int")).alias(c) for c in TEXT_COLS
]).collect()[0].asDict()

print("Values carrying leading/trailing whitespace:")
for c, n in ws_counts.items():
    print(f"  {c:<18} {n if n else 0:>4}")

Values carrying leading/trailing whitespace:
  customer_id           0
  customer_name         3
  product               0
  category              0
  payment_method        0
  city                  3
  state                 0
  order_status          0


In [11]:
# --- Text-format inconsistencies: capitalisation variants ---------------------
# Group by UPPER(TRIM(value)); any group holding more than one spelling is the
# same real-world value written inconsistently.
def case_variants(df, c):
    return (df.filter(F.col(c).isNotNull() & (F.trim(F.col(c)) != ""))
              .groupBy(F.upper(F.trim(F.col(c))).alias("normalised"))
              .agg(F.collect_set(F.col(c)).alias("spellings_found"),
                   F.count("*").alias("rows"))
              .filter(F.size("spellings_found") > 1)
              .withColumn("column", F.lit(c))
              .select("column", "normalised", "spellings_found", "rows"))

variant_report = None
for c in TEXT_COLS:
    part = case_variants(raw, c)
    variant_report = part if variant_report is None else variant_report.unionByName(part)

variant_report.cache()
print(f"Columns/values affected by inconsistent formatting: {variant_report.count()}")
variant_report.orderBy("column", "normalised").show(50, truncate=False)

Columns/values affected by inconsistent formatting: 12
+-------------+---------------+-----------------------------------+----+
|column       |normalised     |spellings_found                    |rows|
+-------------+---------------+-----------------------------------+----+
|category     |BEAUTY         |[beauty, Beauty]                   |138 |
|category     |CLOTHING       |[Clothing, CLOTHING]               |109 |
|category     |ELECTRONICS    |[Electronics, ELECTRONICS]         |104 |
|category     |SPORTS         |[Sports, sports, SPORTS]           |138 |
|city         |COIMBATORE     |[coimbatore, Coimbatore]           |82  |
|city         |DELHI          |[delhi, Delhi, DELHI]              |84  |
|city         |HYDERABAD      |[HYDERABAD,  Hyderabad , Hyderabad]|97  |
|city         |JAIPUR         |[Jaipur, JAIPUR]                   |85  |
|city         |MUMBAI         |[ Mumbai , Mumbai]                 |107 |
|customer_name|AKASH JAIN     |[Akash Jain , Akash Jain]          |2 

In [12]:
# --- Invalid numerical values -------------------------------------------------
# Rules: quantity must be a whole number > 0, unit_price must be > 0,
# discount_percent must lie in [0, 100].
NUM_RULES = {
    "quantity":         lambda v: (v > 0) & (v == F.floor(v)),
    "unit_price":       lambda v: v > 0,
    "discount_percent": lambda v: (v >= 0) & (v <= 100),
}

print(f"{'column':<18}{'not a number':>14}{'out of range':>14}{'valid':>8}")
for c, rule in NUM_RULES.items():
    v = F.col(c).cast("double")
    present = F.col(c).isNotNull() & (F.trim(F.col(c)) != "")
    row = raw.select(
        F.sum((present & v.isNull()).cast("int")).alias("nan"),
        F.sum((v.isNotNull() & ~rule(v)).cast("int")).alias("oor"),
        F.sum((v.isNotNull() & rule(v)).cast("int")).alias("ok"),
    ).collect()[0]
    print(f"{c:<18}{row['nan'] or 0:>14}{row['oor'] or 0:>14}{row['ok'] or 0:>8}")

print("\nOffending values (and how often each occurs):")
for c, rule in NUM_RULES.items():
    v = F.col(c).cast("double")
    bad = (raw.filter(v.isNull() | ~rule(v))
              .groupBy(F.col(c).alias("value")).count()
              .orderBy(F.desc("count")))
    if bad.count():
        print(f"\n  {c}:")
        for r in bad.collect():
            print(f"    {str(r['value']):>12}  x{r['count']}")

column              not a number  out of range   valid
quantity                       0            12     988


unit_price                     0             8     992
discount_percent               0             8     992

Offending values (and how often each occurs):



  quantity:


               0  x6
              -2  x5
              -1  x1

  unit_price:


               0  x4
            -500  x3
            -250  x1

  discount_percent:


              -5  x4
             120  x4


In [13]:
# --- Inconsistent / invalid dates ---------------------------------------------
# The expected format is ISO yyyy-MM-dd. Anything else is either written in a
# different format or is not a date at all.
iso = F.to_date(F.trim(F.col("order_date")), "yyyy-MM-dd")
non_iso = raw.filter(F.col("order_date").isNotNull() & iso.isNull())

print(f"Rows parsing cleanly as yyyy-MM-dd : {TOTAL_ROWS - non_iso.count()}")
print(f"Rows NOT matching yyyy-MM-dd       : {non_iso.count()}")
print("\nThe non-ISO raw values:")
(non_iso.groupBy("order_date").count().orderBy(F.desc("count"))
        .show(truncate=False))

print("Date range of the values that do parse:")
raw.select(F.min(iso).alias("earliest"), F.max(iso).alias("latest")).show()

Rows parsing cleanly as yyyy-MM-dd : 994
Rows NOT matching yyyy-MM-dd       : 6

The non-ISO raw values:


+----------+-----+
|order_date|count|
+----------+-----+
|2025-02-30|1    |
|2025/07/14|1    |
|31/02/2026|1    |
|2026-13-05|1    |
|not-a-date|1    |
|09-15-2026|1    |
+----------+-----+

Date range of the values that do parse:
+----------+----------+
|  earliest|    latest|
+----------+----------+
|2025-01-01|2026-09-30|
+----------+----------+



### 2.2 Consolidated discrepancy summaryEvery number in the table below is carried over from the Spark aggregations runabove — none of them is typed in.

In [14]:
# Gather the detection results into one table, reusing the values Spark computed.
def n_bad(expr):
    return raw.filter(expr).count()

v = F.col("quantity").cast("double")
p = F.col("unit_price").cast("double")
d = F.col("discount_percent").cast("double")
iso_ok = F.to_date(F.trim(F.col("order_date")), "yyyy-MM-dd")

summary = [
    ("exact duplicate rows", exact_dupes, "dropped"),
    ("rows sharing an order_id", dupe_id_rows, "kept + flagged"),
    ("missing customer_name", missing_counts["customer_name"], "filled 'Unknown'"),
    ("missing city", missing_counts["city"], "filled 'Unknown'"),
    ("missing payment_method", missing_counts["payment_method"], "filled with mode"),
    ("values with stray whitespace", sum(n or 0 for n in ws_counts.values()), "trimmed"),
    ("values with a case variant", variant_report.count(), "folded to dominant spelling"),
    ("quantity not a whole number > 0", n_bad(v.isNull() | (v <= 0) | (v != F.floor(v))), "imputed (median)"),
    ("unit_price not > 0", n_bad(p.isNull() | (p <= 0)), "imputed (product median)"),
    ("discount_percent outside 0-100", n_bad(d.isNull() | (d < 0) | (d > 100)), "reset to 0"),
    ("order_date not in yyyy-MM-dd", n_bad(iso_ok.isNull()), "re-parsed, or NULL if impossible"),
]

print(f"{'discrepancy':<34}{'rows':>6}   treatment in Silver")
print("-" * 80)
for label, count, action in summary:
    print(f"{label:<34}{count:>6}   {action}")
print("-" * 80)
print("Only the exact duplicates are removed; everything else is repaired in place.")

discrepancy                         rows   treatment in Silver
--------------------------------------------------------------------------------
exact duplicate rows                  10   dropped
rows sharing an order_id              18   kept + flagged
missing customer_name                 10   filled 'Unknown'
missing city                           8   filled 'Unknown'
missing payment_method                 8   filled with mode
values with stray whitespace           6   trimmed
values with a case variant            12   folded to dominant spelling
quantity not a whole number > 0       12   imputed (median)
unit_price not > 0                     8   imputed (product median)
discount_percent outside 0-100         8   reset to 0
order_date not in yyyy-MM-dd           6   re-parsed, or NULL if impossible
--------------------------------------------------------------------------------
Only the exact duplicates are removed; everything else is repaired in place.


## 3. Basic preprocessing → Silver layerSix steps, applied in this order:1. remove exact duplicate rows2. trim whitespace and standardise capitalisation in text fields3. validate the numeric fields (flag what is wrong before changing anything)4. parse `order_date` to a single ISO date type5. handle missing values and repair the invalid numbers6. record every repair in `dq_flags`Steps 3 and 5 are deliberately split: the *flags* are computed against theoriginal values, so the audit trail survives the imputation that follows.

In [15]:
# --- Step 1: remove exact duplicate rows --------------------------------------
df = raw.dropDuplicates()
print(f"rows before: {TOTAL_ROWS}")
print(f"rows after dropDuplicates(): {df.count()}  (removed {TOTAL_ROWS - df.count()})")

rows before: 1000
rows after dropDuplicates(): 990  (removed 10)


In [16]:
# --- Step 2a: trim whitespace in every text column ----------------------------
# Also collapse any run of internal whitespace to a single space, and turn
# now-empty strings into proper NULLs so one missing-value rule covers them all.
for c in TEXT_COLS:
    cleaned = F.regexp_replace(F.trim(F.col(c)), r"\s+", " ")
    df = df.withColumn(c, F.when(cleaned == "", None).otherwise(cleaned))

df.select("customer_name", "city", "category", "payment_method").show(5, truncate=False)

+--------------+---------+---------------+--------------+
|customer_name |city     |category       |payment_method|
+--------------+---------+---------------+--------------+
|Sanjay Menon  |Chennai  |Clothing       |Debit Card    |
|Sakshi Bose   |Hyderabad|Clothing       |Debit Card    |
|Arjun Patel   |Mumbai   |Home Appliances|Debit Card    |
|Pranav Sharma |Mumbai   |Sports         |Net Banking   |
|Lakshmi Pillai|Lucknow  |Beauty         |Debit Card    |
+--------------+---------+---------------+--------------+
only showing top 5 rows



In [17]:
# --- Step 2b: standardise capitalisation (data-driven, nothing hard-coded) ----
# For each UPPER(value) group, the most frequent spelling present in the data
# wins and is applied to the whole group. So `beauty`/`BEAUTY` become `Beauty`,
# while a legitimately mixed-case value such as `Cash on Delivery` is left alone
# — which a blanket initcap() would have mangled into `Cash On Delivery`.
#
# Spark derives the mapping (groupBy + window); it is then collected, printed for
# transparency, and applied with replace(). Collecting is safe here because the
# mapping holds only the handful of misspelt variants, not the data itself.
def case_mapping(df, c):
    counts = (df.filter(F.col(c).isNotNull())
                .groupBy(F.upper(F.col(c)).alias("_key"), F.col(c).alias("_variant"))
                .count())
    w = Window.partitionBy("_key").orderBy(F.desc("count"), F.asc("_variant"))
    ranked = counts.withColumn("_rank", F.row_number().over(w))
    winner = (ranked.filter(F.col("_rank") == 1)
                    .select("_key", F.col("_variant").alias("_canonical")))
    losers = (ranked.join(winner, on="_key", how="inner")
                    .filter(F.col("_variant") != F.col("_canonical"))
                    .select("_variant", "_canonical", "count"))
    return {r["_variant"]: r["_canonical"] for r in losers.collect()}

CASE_COLS = ["category", "payment_method", "city", "state", "order_status", "product"]

for c in CASE_COLS:
    mapping = case_mapping(df, c)
    if mapping:
        print(f"{c}:")
        for old, new in sorted(mapping.items()):
            print(f"    {old!r:<20} -> {new!r}")
        df = df.replace(mapping, subset=[c])
    else:
        print(f"{c}: already consistent")

df = df.cache()
print()
for c in CASE_COLS:
    n = df.select(F.countDistinct(c)).collect()[0][0]
    print(f"  {c:<18} distinct values after standardising: {n}")

category:
    'CLOTHING'           -> 'Clothing'
    'ELECTRONICS'        -> 'Electronics'
    'SPORTS'             -> 'Sports'
    'beauty'             -> 'Beauty'
    'sports'             -> 'Sports'
payment_method: already consistent


city:
    'DELHI'              -> 'Delhi'
    'HYDERABAD'          -> 'Hyderabad'
    'JAIPUR'             -> 'Jaipur'
    'coimbatore'         -> 'Coimbatore'
    'delhi'              -> 'Delhi'
state: already consistent


order_status: already consistent
product: already consistent



  category           distinct values after standardising: 8
  payment_method     distinct values after standardising: 5
  city               distinct values after standardising: 11
  state              distinct values after standardising: 9
  order_status       distinct values after standardising: 5


  product            distinct values after standardising: 57


In [18]:
print("Category values after standardising:")
df.groupBy("category").count().orderBy(F.desc("count")).show(truncate=False)
print("City values after standardising:")
df.groupBy("city").count().orderBy(F.desc("count")).show(truncate=False)

Category values after standardising:
+---------------+-----+
|category       |count|
+---------------+-----+
|Sports         |138  |
|Beauty         |137  |
|Grocery        |134  |
|Furniture      |129  |
|Books          |127  |
|Home Appliances|113  |
|Clothing       |108  |
|Electronics    |104  |
+---------------+-----+

City values after standardising:
+----------+-----+
|city      |count|
+----------+-----+
|Mumbai    |105  |
|Vijayawada|104  |
|Hyderabad |97   |
|Bengaluru |90   |
|Lucknow   |89   |
|Chennai   |88   |
|Delhi     |83   |
|Jaipur    |83   |
|Pune      |81   |
|Kolkata   |81   |
|Coimbatore|81   |
|NULL      |8    |
+----------+-----+



In [19]:
# --- Step 3: validate the numeric fields (flags only, no values changed yet) --
qty   = F.col("quantity").cast("double")
price = F.col("unit_price").cast("double")
disc  = F.col("discount_percent").cast("double")

df = (df
      .withColumn("_qty_num",   qty)
      .withColumn("_price_num", price)
      .withColumn("_disc_num",  disc)
      .withColumn("_qty_bad",   ~(qty.isNotNull()   & (qty > 0) & (qty == F.floor(qty))))
      .withColumn("_price_bad", ~(price.isNotNull() & (price > 0)))
      .withColumn("_disc_bad",  ~(disc.isNotNull()  & (disc >= 0) & (disc <= 100))))

df.select(
    F.sum(F.col("_qty_bad").cast("int")).alias("quantity_to_repair"),
    F.sum(F.col("_price_bad").cast("int")).alias("unit_price_to_repair"),
    F.sum(F.col("_disc_bad").cast("int")).alias("discount_to_repair"),
).show()

+------------------+--------------------+------------------+
|quantity_to_repair|unit_price_to_repair|discount_to_repair|
+------------------+--------------------+------------------+
|                12|                   8|                 8|
+------------------+--------------------+------------------+



In [20]:
# --- Step 4: one consistent date format --------------------------------------
# Several layouts are attempted in priority order. The first one that yields a
# real calendar date wins; a value that satisfies none (`not-a-date`, or the
# impossible 31 February) becomes NULL rather than a guess.
#
# Note on ordering: dd-MM-yyyy is tried before MM-dd-yyyy, so `09-15-2026` fails
# the first (month 15 does not exist) and is correctly read by the second.
DATE_FORMATS = ["yyyy-MM-dd", "yyyy/MM/dd", "dd/MM/yyyy",
                "dd-MM-yyyy", "MM-dd-yyyy", "yyyy.MM.dd", "dd-MMM-yyyy"]

parsed_date = F.coalesce(*[F.to_date(F.col("order_date"), fmt) for fmt in DATE_FORMATS])

df = (df.withColumn("_raw_date", F.col("order_date"))
        .withColumn("order_date", parsed_date)
        .withColumn("_date_bad", F.col("order_date").isNull()))

print("Values that could not be parsed into any accepted format:")
(df.filter(F.col("_date_bad"))
   .groupBy("_raw_date").count().show(truncate=False))

print("Values that were in the wrong format but were recovered:")
(df.filter(~F.col("_date_bad")
           & (F.col("_raw_date") != F.date_format("order_date", "yyyy-MM-dd")))
   .select("order_id", "_raw_date", "order_date").show(truncate=False))

Values that could not be parsed into any accepted format:
+----------+-----+
|_raw_date |count|
+----------+-----+
|2025-02-30|1    |
|31/02/2026|1    |
|not-a-date|1    |
|2026-13-05|1    |
+----------+-----+

Values that were in the wrong format but were recovered:
+--------+----------+----------+
|order_id|_raw_date |order_date|
+--------+----------+----------+
|100969  |09-15-2026|2026-09-15|
|100358  |2025/07/14|2025-07-14|
+--------+----------+----------+



In [21]:
# --- Step 5: handle missing values and repair the invalid numbers -------------
# Chosen methods, and why:
#   customer_name    -> "Unknown"       : cannot be derived from anything else.
#   city             -> "Unknown"       : the state is known, but several cities
#                                         share a state, so picking one would be
#                                         an invention rather than a repair.
#   payment_method   -> modal value      : a single dominant category makes the
#                                         mode the least distorting choice.
#   quantity         -> overall median   : robust to the outliers present.
#   unit_price       -> median price of the SAME product, falling back to the
#                       overall median — price depends strongly on the product,
#                       so a per-product median is far closer to the truth.
#   discount_percent -> 0               : "no discount" is the safe reading of a
#                                         negative or >100 % value.

modal_payment = (df.filter(F.col("payment_method").isNotNull())
                   .groupBy("payment_method").count()
                   .orderBy(F.desc("count")).first()["payment_method"])

median_qty = df.filter(~F.col("_qty_bad")).approxQuantile("_qty_num", [0.5], 0.0)[0]
median_price = df.filter(~F.col("_price_bad")).approxQuantile("_price_num", [0.5], 0.0)[0]

product_median_price = (df.filter(~F.col("_price_bad"))
                          .groupBy("product")
                          .agg(F.expr("percentile_approx(_price_num, 0.5)").alias("_prod_median")))

print(f"modal payment_method : {modal_payment}")
print(f"median quantity      : {median_qty}")
print(f"median unit_price    : {median_price}")

df = (df.join(F.broadcast(product_median_price), on="product", how="left")
        .withColumn("_name_bad",    F.col("customer_name").isNull())
        .withColumn("_city_bad",    F.col("city").isNull())
        .withColumn("_payment_bad", F.col("payment_method").isNull())
        .withColumn("customer_name",
                    F.coalesce(F.col("customer_name"), F.lit("Unknown")))
        .withColumn("city", F.coalesce(F.col("city"), F.lit("Unknown")))
        .withColumn("payment_method",
                    F.coalesce(F.col("payment_method"), F.lit(modal_payment)))
        .withColumn("quantity",
                    F.when(F.col("_qty_bad"), F.lit(median_qty))
                     .otherwise(F.col("_qty_num")).cast("int"))
        .withColumn("unit_price",
                    F.when(F.col("_price_bad"),
                           F.coalesce(F.col("_prod_median"), F.lit(median_price)))
                     .otherwise(F.col("_price_num")).cast("double"))
        .withColumn("discount_percent",
                    F.when(F.col("_disc_bad"), F.lit(0.0))
                     .otherwise(F.col("_disc_num")).cast("double"))
        .drop("_prod_median"))

modal payment_method : Cash on Delivery
median quantity      : 5.0
median unit_price    : 1850.0


In [22]:
# --- Step 6: record every repair in an auditable dq_flags column -------------
# concat_ws skips NULL inputs, so each when() that does not fire simply
# contributes nothing.
dup_id = F.count("*").over(Window.partitionBy("order_id")) > 1

df = (df.withColumn("_dup_id", dup_id)
        .withColumn("dq_flags", F.concat_ws(";",
            F.when(F.col("_name_bad"),    F.lit("NAME_FILLED")),
            F.when(F.col("_city_bad"),    F.lit("CITY_FILLED")),
            F.when(F.col("_payment_bad"), F.lit("PAYMENT_FILLED")),
            F.when(F.col("_qty_bad"),     F.lit("QUANTITY_IMPUTED")),
            F.when(F.col("_price_bad"),   F.lit("UNIT_PRICE_IMPUTED")),
            F.when(F.col("_disc_bad"),    F.lit("DISCOUNT_RESET_0")),
            F.when(F.col("_date_bad"),    F.lit("DATE_UNPARSEABLE")),
            F.when(F.col("_dup_id"),      F.lit("DUPLICATE_ORDER_ID")),
        ))
        .withColumn("dq_flags",
                    F.when(F.col("dq_flags") == "", F.lit("OK"))
                     .otherwise(F.col("dq_flags"))))

silver = df.select(
    "order_id", "customer_id", "customer_name", "product", "category",
    "quantity", "unit_price", "discount_percent", "payment_method",
    "city", "state", "order_date", "order_status", "dq_flags",
).orderBy("order_id")

silver.cache()
SILVER_ROWS = silver.count()

print("Silver schema:")
silver.printSchema()
print(f"Silver rows: {SILVER_ROWS}")
silver.show(10, truncate=False)

Silver schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- dq_flags: string (nullable = false)

Silver rows: 990
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+--------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|dq_flags|
+--------

In [23]:
# Flag distribution — how many rows were touched, and how.
print(f"Rows needing no repair at all: "
      f"{silver.filter(F.col('dq_flags') == 'OK').count()} / {SILVER_ROWS}")
print("\nRepairs applied, by flag:")
(silver.filter(F.col("dq_flags") != "OK")
       .select(F.explode(F.split("dq_flags", ";")).alias("flag"))
       .groupBy("flag").count().orderBy(F.desc("count"))
       .show(truncate=False))

print("A sample of repaired rows:")
silver.filter(F.col("dq_flags") != "OK").show(15, truncate=False)

Rows needing no repair at all: 916 / 990

Repairs applied, by flag:
+------------------+-----+
|flag              |count|
+------------------+-----+
|DUPLICATE_ORDER_ID|16   |
|QUANTITY_IMPUTED  |12   |
|NAME_FILLED       |10   |
|UNIT_PRICE_IMPUTED|8    |
|DISCOUNT_RESET_0  |8    |
|PAYMENT_FILLED    |8    |
|CITY_FILLED       |8    |
|DATE_UNPARSEABLE  |4    |
+------------------+-----+

A sample of repaired rows:


+--------+-----------+---------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+------------------+
|order_id|customer_id|customer_name  |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|dq_flags          |
+--------+-----------+---------------+----------------------+---------------+--------+----------+----------------+----------------+----------+--------------+----------+------------+------------------+
|100013  |C0196      |Unknown        |Competitive Exam Guide|Books          |3       |660.0     |12.0            |Credit Card     |Chennai   |Tamil Nadu    |2025-08-16|Delivered   |NAME_FILLED       |
|100034  |C0192      |Divya Sharma   |Yoga Mat              |Sports         |7       |1720.0    |0.0             |Cash on Delivery|Coimbatore|Tamil Nadu    |2025-08-19|Delivered   |DISCOUNT_RESET_

## 4. Write the Silver layerParquet is the primary format (typed, compressed, the normal choice for a Silverlayer). A CSV copy is written alongside it for human inspection and for theassignment submission; `coalesce(1)` forces a single part file so the result isone readable CSV rather than a directory of fragments.

In [24]:
silver.write.mode("overwrite").parquet(SILVER_PARQUET)
(silver.coalesce(1).write.mode("overwrite")
       .option("header", True)
       .option("dateFormat", "yyyy-MM-dd")
       .csv(SILVER_CSV_DIR))

# Promote the single Spark part file to a plainly-named CSV.
part = glob.glob(f"{SILVER_CSV_DIR}/part-*.csv")[0]
os.makedirs("silver", exist_ok=True)
shutil.copyfile(part, "silver/sales_silver.csv")

print("Written:")
print("  ", SILVER_PARQUET, "(parquet)")
print("  ", SILVER_CSV_DIR, "(csv part files)")
print("   silver/sales_silver.csv", f"({os.path.getsize('silver/sales_silver.csv')} bytes)")

Written:
   data_lake/silver/preprocessed_sales (parquet)
   data_lake/silver/preprocessed_sales_csv (csv part files)
   silver/sales_silver.csv (115539 bytes)


In [25]:
# Final data lake layout.
for root, dirs, files in sorted(os.walk("data_lake")):
    depth = root.count(os.sep)
    print("    " * depth + os.path.basename(root) + "/")
    for f in sorted(files):
        if f.startswith(".") or f.endswith(".crc"):
            continue
        print("    " * (depth + 1) + f)

data_lake/
    bronze/
        sales.csv
    silver/
        preprocessed_sales/
            _SUCCESS
            part-00000-1847c567-95fb-48cc-a624-511195d4dd7c-c000.snappy.parquet
            part-00001-1847c567-95fb-48cc-a624-511195d4dd7c-c000.snappy.parquet
            part-00002-1847c567-95fb-48cc-a624-511195d4dd7c-c000.snappy.parquet
            part-00003-1847c567-95fb-48cc-a624-511195d4dd7c-c000.snappy.parquet
            part-00004-1847c567-95fb-48cc-a624-511195d4dd7c-c000.snappy.parquet
            part-00005-1847c567-95fb-48cc-a624-511195d4dd7c-c000.snappy.parquet
            part-00006-1847c567-95fb-48cc-a624-511195d4dd7c-c000.snappy.parquet
            part-00007-1847c567-95fb-48cc-a624-511195d4dd7c-c000.snappy.parquet
        preprocessed_sales_csv/
            _SUCCESS
            part-00000-f9293ea9-c4a5-4bff-ab91-4e0c7915255f-c000.csv


## 5. Verification

In [26]:
# --- Row count before and after ----------------------------------------------
print(f"Bronze rows (before preprocessing) : {TOTAL_ROWS}")
print(f"Silver rows (after preprocessing)  : {SILVER_ROWS}")
print(f"Rows removed                       : {TOTAL_ROWS - SILVER_ROWS} "
      f"(exact duplicates only)")
print(f"Records retained                   : {100 * SILVER_ROWS / TOTAL_ROWS:.1f}%")

Bronze rows (before preprocessing) : 1000
Silver rows (after preprocessing)  : 990
Rows removed                       : 10 (exact duplicates only)
Records retained                   : 99.0%


In [27]:
# --- Read the Silver data back with Spark ------------------------------------
silver_back = spark.read.parquet(SILVER_PARQUET)
print(f"Rows read back from Silver (parquet): {silver_back.count()}")
silver_back.printSchema()
silver_back.orderBy("order_id").show(10, truncate=False)

Rows read back from Silver (parquet): 990
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount_percent: double (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)
 |-- dq_flags: string (nullable = true)

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+--------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|dq_flags|
+--

In [28]:
# Read the CSV copy back too, to confirm it is well-formed and round-trips.
csv_back = spark.read.csv("silver/sales_silver.csv", header=True, inferSchema=True)
print(f"Rows read back from silver/sales_silver.csv: {csv_back.count()}")
csv_back.show(5, truncate=False)

Rows read back from silver/sales_silver.csv: 990
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+--------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|dq_flags|
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+--------+
|100001  |C0256      |Keerthi Mishra|Saree        |Clothing|2       |6710.0    |20.0            |Credit Card   |Mumbai    |Maharashtra   |2026-01-01|Delivered   |OK      |
|100002  |C0042      |Arjun Singh   |T-Shirt      |Clothing|6       |680.0     |5.0             |Debit Card    |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |OK      |
|100003  |C0348      |Keerthi Rao   |Hoodie       |Clothing|6       |3080.0    |12.0       

In [29]:
# --- Post-conditions ----------------------------------------------------------
checks = {}
checks["no exact duplicate rows"] = silver_back.count() == silver_back.distinct().count()
checks["no missing customer_name"] = silver_back.filter(F.col("customer_name").isNull()).count() == 0
checks["no missing city"] = silver_back.filter(F.col("city").isNull()).count() == 0
checks["no missing payment_method"] = silver_back.filter(F.col("payment_method").isNull()).count() == 0
checks["quantity is a positive integer"] = silver_back.filter(
    F.col("quantity").isNull() | (F.col("quantity") <= 0)).count() == 0
checks["unit_price > 0"] = silver_back.filter(
    F.col("unit_price").isNull() | (F.col("unit_price") <= 0)).count() == 0
checks["discount_percent within 0-100"] = silver_back.filter(
    F.col("discount_percent").isNull()
    | (F.col("discount_percent") < 0) | (F.col("discount_percent") > 100)).count() == 0
checks["order_date is a DateType column"] = dict(silver_back.dtypes)["order_date"] == "date"
checks["no stray whitespace in text fields"] = silver_back.filter(
    F.expr(" OR ".join(f"{c} <> trim({c})" for c in TEXT_COLS))).count() == 0
checks["no case-variant duplicates in category"] = (
    silver_back.select(F.countDistinct("category")).collect()[0][0]
    == silver_back.select(F.countDistinct(F.upper("category"))).collect()[0][0])
checks["Bronze layer untouched"] = sha256(BRONZE_FILE) == BRONZE_SHA_BEFORE

for name, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
assert all(checks.values()), "a post-condition failed"
print("\nAll post-conditions passed.")

  [PASS] no exact duplicate rows
  [PASS] no missing customer_name
  [PASS] no missing city
  [PASS] no missing payment_method
  [PASS] quantity is a positive integer
  [PASS] unit_price > 0
  [PASS] discount_percent within 0-100
  [PASS] order_date is a DateType column
  [PASS] no stray whitespace in text fields
  [PASS] no case-variant duplicates in category
  [PASS] Bronze layer untouched

All post-conditions passed.


In [30]:
# Rows intentionally left imperfect, as the brief asks: a NULL order_date is
# honest about an unparseable input, and same-order_id rows that differ in
# content are kept rather than silently collapsed.
print("Rows with an unresolved date (kept on purpose):")
silver_back.filter(F.col("order_date").isNull()).show(truncate=False)

print("Rows sharing an order_id but differing in content (kept on purpose):")
(silver_back.filter(F.col("dq_flags").contains("DUPLICATE_ORDER_ID"))
            .orderBy("order_id").show(truncate=False))

Rows with an unresolved date (kept on purpose):
+--------+-----------+-------------+-------------+---------+--------+----------+----------------+----------------+---------+-------------+----------+------------+----------------+
|order_id|customer_id|customer_name|product      |category |quantity|unit_price|discount_percent|payment_method  |city     |state        |order_date|order_status|dq_flags        |
+--------+-----------+-------------+-------------+---------+--------+----------+----------------+----------------+---------+-------------+----------+------------+----------------+
|100634  |C0262      |Tanvi Mehta  |Atta 10kg    |Grocery  |10      |560.0     |15.0            |Cash on Delivery|Hyderabad|Telangana    |NULL      |Processing  |DATE_UNPARSEABLE|
|100637  |C0020      |Aarav Patel  |Novel        |Books    |4       |610.0     |20.0            |Net Banking     |Chennai  |Tamil Nadu   |NULL      |Delivered   |DATE_UNPARSEABLE|
|100097  |C0244      |Shreya Rao   |Sofa         |Fu

In [31]:
# Date range after cleaning, for a final sanity check.
silver_back.select(
    F.min("order_date").alias("earliest_order"),
    F.max("order_date").alias("latest_order"),
    F.count("order_date").alias("dates_present"),
    F.sum(F.col("order_date").isNull().cast("int")).alias("dates_null"),
).show()

+--------------+------------+-------------+----------+
|earliest_order|latest_order|dates_present|dates_null|
+--------------+------------+-------------+----------+
|    2025-01-01|  2026-09-30|          986|         4|
+--------------+------------+-------------+----------+



## 6. Summary of preprocessing performed**Bronze → Silver, step by step**1. **Exact duplicates removed** — `dropDuplicates()` over all 13 columns.2. **Whitespace normalised** — every text column trimmed, internal runs of   whitespace collapsed to one space, and strings left empty converted to `NULL`   so a single missing-value rule covers blanks and `NULL`s alike.3. **Capitalisation standardised** — values were grouped by `UPPER(value)` and   each group folded to its most frequent spelling in the data. This is derived   from the dataset rather than from a hard-coded list, and unlike a blanket   `initcap()` it leaves legitimately mixed-case values such as   `Cash on Delivery` alone.4. **Numeric validation** — `quantity` must be a whole number > 0, `unit_price`   must be > 0, `discount_percent` must fall in 0–100. Violations were repaired,   not deleted: `quantity` from the overall median, `unit_price` from the median   price of the *same product* (overall median as fallback), and an out-of-range   `discount_percent` reset to 0.5. **Dates unified to one ISO format** — `order_date` is parsed against several   candidate layouts in priority order and stored as a true `DateType`. Values   that match no layout, or that name a day which does not exist, are set to   `NULL` instead of being guessed at.6. **Missing values filled** — `customer_name` and `city` with `"Unknown"`   (neither can be honestly derived from the other columns), `payment_method`   with the modal value computed by Spark.7. **Every change recorded** — `dq_flags` names each repair a row received, so   an imputed value is never mistaken for an observed one. Clean rows read `OK`.**What was deliberately *not* done**The brief asks for useful records to be preserved rather than for a perfectlyclean dataset, so:- **Rows sharing an `order_id` but differing elsewhere were kept.** They are not  duplicates — they differ in content — and only one of the two can be wrong.  Choosing which to delete would need business rules the assignment does not  supply, so both are kept and flagged `DUPLICATE_ORDER_ID`.- **Unparseable dates leave a `NULL`, not a substitute.** The rest of each such  record is perfectly good; replacing the date with a median or a placeholder  would fabricate information, and deleting the row would discard a real sale.- **Legitimate outliers were left alone.** A `unit_price` of 81,010 and a  `quantity` of 10 are plausible for furniture and groceries respectively, so  only values violating a stated rule were touched.**Layers**| Layer | Path | Format | Rows ||---|---|---|---|| Bronze | `data_lake/bronze/sales.csv` | raw CSV, unmodified (SHA-256 re-verified above) | 1,000 || Silver | `data_lake/silver/preprocessed_sales/` | Parquet | see section 5 || Silver | `silver/sales_silver.csv` | single-file CSV, for submission | see section 5 |

In [32]:
spark.stop()
print("SparkSession stopped. Notebook complete.")

SparkSession stopped. Notebook complete.
